In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import pickle
import json
import os

# ---- Step 1: Load data ----
matches = pd.read_csv('data/new_matches.csv')
matches = matches.dropna(subset=['winner'])
print(f"Total matches: {len(matches)}")

# ---- Step 1.5: Clean all text columns ----
for col in ['team1', 'team2', 'toss_winner', 'winner']:
    matches[col] = matches[col].str.strip()
    
    
# Remove abandoned matches and non-team winners
current_teams = [
    'Mumbai Indians',
    'Chennai Super Kings',
    'Kolkata Knight Riders',
    'Royal Challengers Bengaluru',
    'Sunrisers Hyderabad',
    'Delhi Capitals',
    'Rajasthan Royals',
    'Punjab Kings',
    'Gujarat Titans',
    'Lucknow Super Giants'
]

# Keep only rows where winner is a valid team
matches = matches[matches['winner'].isin(current_teams)]

print(f"After removing abandoned matches: {len(matches)}")
print(f"Unique winners: {sorted(matches['winner'].unique())}")


# Fix short names in toss_winner
toss_name_fix = {
    'MI'   : 'Mumbai Indians',
    'CSK'  : 'Chennai Super Kings',
    'KKR'  : 'Kolkata Knight Riders',
    'RCB'  : 'Royal Challengers Bengaluru',
    'SRH'  : 'Sunrisers Hyderabad',
    'DC'   : 'Delhi Capitals',
    'RR'   : 'Rajasthan Royals',
    'PBKS' : 'Punjab Kings',
    'GT'   : 'Gujarat Titans',
    'LSG'  : 'Lucknow Super Giants',
}
matches['toss_winner'] = matches['toss_winner'].replace(toss_name_fix)

print("Unique toss winners:")
print(sorted(matches['toss_winner'].dropna().unique()))

# ---- Step 2: Add features ----
home_grounds = {
    'Mumbai Indians'              : 'Wankhede',
    'Chennai Super Kings'         : 'Chidambaram',
    'Kolkata Knight Riders'       : 'Eden Gardens',
    'Royal Challengers Bengaluru' : 'Chinnaswamy',
    'Sunrisers Hyderabad'         : 'Rajiv Gandhi',
    'Delhi Capitals'              : 'Feroz Shah',
    'Rajasthan Royals'            : 'Sawai Mansingh',
    'Punjab Kings'                : 'Punjab Cricket',
    'Gujarat Titans'              : 'Narendra Modi',
    'Lucknow Super Giants'        : 'Ekana'
}

def is_home(team, venue):
    keyword = home_grounds.get(team, '')
    return 1 if keyword.lower() in str(venue).lower() else 0

matches['team1_is_home'] = matches.apply(
    lambda x: is_home(x['team1'], x['venue']), axis=1
)
matches['team2_is_home'] = matches.apply(
    lambda x: is_home(x['team2'], x['venue']), axis=1
)

win_counts   = matches['winner'].value_counts()
total_counts = matches['team1'].value_counts() + matches['team2'].value_counts()
win_rate     = (win_counts / total_counts).fillna(0)

matches['team1_win_rate'] = matches['team1'].map(win_rate).fillna(0)
matches['team2_win_rate'] = matches['team2'].map(win_rate).fillna(0)

matches['toss_winner_won'] = (
    matches['toss_winner'] == matches['winner']
).astype(int)

print("\nFeatures added!")

# ---- Step 3: Filter toss data ----
matches = matches.dropna(subset=['toss_winner', 'toss_decision'])
print(f"After toss filter: {len(matches)} matches")

# ---- Step 4: Encode separately ----
all_teams = sorted(matches['team1'].unique().tolist())

le_team    = LabelEncoder()
le_toss    = LabelEncoder()
le_winner  = LabelEncoder()

le_team.fit(all_teams)
le_toss.fit(['bat', 'field'])
le_winner.fit(current_teams)

print("\nTeam encoding:")
for i, team in enumerate(le_team.classes_):
    print(f"  {team} → {i}")

matches_encoded = matches.copy()
matches_encoded['team1']         = le_team.transform(matches['team1'].astype(str))
matches_encoded['team2']         = le_team.transform(matches['team2'].astype(str))
matches_encoded['toss_winner']   = le_team.transform(matches['toss_winner'].astype(str))
matches_encoded['toss_decision'] = le_toss.transform(matches['toss_decision'].astype(str))
matches_encoded['winner']        = le_winner.transform(matches['winner'].astype(str))

print("\nEncoding done!")

# ---- Step 5: Features and target ----
features = [
    'team1',
    'team2',
    'toss_winner',
    'toss_decision',
    'team1_is_home',
    'team2_is_home',
    'team1_win_rate',
    'team2_win_rate',
    'toss_winner_won'
]

X = matches_encoded[features]
y = matches_encoded['winner']

print(f"\nTraining on {len(X)} matches")

# ---- Step 6: Train test split ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---- Step 7: Train Random Forest ----
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42
)
model.fit(X_train, y_train)

accuracy = accuracy_score(y_test, model.predict(X_test))
print(f"\nAccuracy: {accuracy:.2f}")

# ---- Step 8: Feature importance ----
import matplotlib.pyplot as plt

importances = pd.Series(
    model.feature_importances_,
    index=features
).sort_values()

importances.plot(kind='barh', figsize=(8,5))
plt.title("Feature Importance")
plt.tight_layout()
plt.show()

# ---- Step 9: Save everything ----
os.makedirs('models', exist_ok=True)

with open('models/match_winner_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('models/team_encoder.pkl', 'wb') as f:
    pickle.dump(le_team, f)

with open('models/toss_encoder.pkl', 'wb') as f:
    pickle.dump(le_toss, f)

winner_classes = le_winner.classes_.tolist()
with open('models/winner_classes.json', 'w') as f:
    json.dump(winner_classes, f)

win_rate_dict = win_rate.to_dict()
with open('models/win_rate.json', 'w') as f:
    json.dump(win_rate_dict, f)

print("\nAll files saved:")
print("✅ match_winner_model.pkl")
print("✅ team_encoder.pkl")
print("✅ toss_encoder.pkl")
print("✅ winner_classes.json")
print("✅ win_rate.json")

Total matches: 1181
Unique toss winners:
['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Titans', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Punjab Kings', 'Rajasthan Royals', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']

Features added!
After toss filter: 1037 matches

Team encoding:
  Chennai Super Kings → 0
  Delhi Capitals → 1
  Gujarat Titans → 2
  Kolkata Knight Riders → 3
  Lucknow Super Giants → 4
  Mumbai Indians → 5
  Punjab Kings → 6
  Rajasthan Royals → 7
  Royal Challengers Bengaluru → 8
  Sunrisers Hyderabad → 9


ValueError: y contains previously unseen labels: np.str_('Match Abandoned Due To Rain')